### 39. 组合总和
给你一个 无重复元素 的整数数组 candidates 和一个目标整数 target ，找出 candidates 中可以使数字和为目标数 target 的 所有 不同组合 ，并以列表形式返回。你可以按 任意顺序 返回这些组合。

candidates 中的 同一个 数字可以 无限制重复被选取 。如果至少一个数字的被选数量不同，则两种组合是不同的。 

对于给定的输入，保证和为 target 的不同组合数少于 150 个。

示例 1：

输入：candidates = [2,3,6,7], target = 7

输出：[[2,2,3],[7]]

解释：
2 和 3 可以形成一组候选，2 + 2 + 3 = 7 。注意 2 可以使用多次。
7 也是一个候选， 7 = 7 。

示例 2：

输入: candidates = [2,3,5], target = 8

输出: [[2,2,2,2],[2,3,3],[3,5]]

#### 1. 回溯 选或者不选
用 dfs(i,left) 来回溯，设当前枚举到 candidates[i]，剩余要选的元素之和为 left，按照选或不选当前元素分类讨论：

- 不选 candidates[i]：递归到 dfs(i+1,left)，继续判断选不选 i+1 的数。
- 选 candidates[i]：递归到 dfs(i,left−candidates[i])。注意 i 不变，表示在下次递归中可以继续选 candidates[i]，重新判断选不选 i 的数。

作者：灵茶山艾府
链接：https://leetcode.cn/problems/combination-sum/solutions/


In [ ]:
from typing import List
class Solution:
    def combinationSum(self, candidates: List[int], target: int) -> List[List[int]]:
        # 选或不选
        candidates.sort() # 优化，先排序，剪枝时可以直接跳过后续元素
        ans = [] # 记录所有组合
        path = [] # 记录当前组合

        def dfs(i: int, left: int) -> None: # 当前选择位置，剩余数
            if left == 0: # 所需余数为0，达到target，加入结果中
                ans.append(path.copy()) # 记录当前组合
                return
            
            if i == len(candidates) or left < candidates[i]: # 超出范围或剩余数不足，无法继续
                return
            
            # 不选择当前元素，继续选择构建后续路径
            dfs(i + 1, left)
            # 选择当前元素，加入路径中
            path.append(candidates[i])
            dfs(i, left - candidates[i]) # 可重复选择，继续判断选不选当前元素
            path.pop() # 此位置判断结束，恢复现场

        dfs(0,target)
        return ans
            

#### 2.回溯 + 剪枝：
回溯全排序 出现的问题在于：在可重复元素情况下，搜索过程是区分选择顺序的，然而子集不区分选择顺序。如下图所示，先选 4 后选 5 与先选 5 后选 4 是两个不同的分支，但两者对应同一个子集。[4,5] 和 [5,4] 是相同的组合。

**考虑在搜索过程中通过剪枝进行去重**。

为实现该剪枝，我们初始化变量 start ，用于指示遍历起点。当做出选择 xi​ 后，设定下一轮从索引 i 开始遍历。这样做就可以让选择序列满足 i1 ≤ i2​ ≤⋯≤ im​ ，从而保证子集唯一。

除此之外，我们还对代码进行了两项优化：

- 在开启搜索前，先将数组 nums 排序。在遍历所有选择时，当子集和超过 target 时直接结束循环，因为后边的元素更大，其子集和都一定会超过 target 。
- 省去元素和变量 total，通过在 target 上执行减法来统计元素和，当 target 等于 0 时记录解。
- 
仍然用 dfs(i,left) 来回溯，设当前枚举到 candidates[i]，剩余要选的元素之和为 left，考虑枚举下个元素是谁：
- 在 [i,n−1] 中枚举要填在 path 中的元素 candidates[j]，然后递归到 dfs(j,left−candidates[j])。注意这里是递归到 j 不是 j+1，表示 candidates[j] 可以重复选取。

In [ ]:
class Solution:
    def combinationSum(self, candidates: List[int], target: int) -> List[List[int]]:
        candidates.sort() # 优化，先排序，剪枝时可以直接跳过后续元素
        res = []
        path = []

        # 选择余下的哪一个
        def dfs(i: int, left: int) -> None: # 当前选择位置，剩余数
            if left == 0: # 找到一个结果
                res.append(path.copy())
                return
            
            # 枚举选哪一个，从当前位置 不回头
            for j in range(i, len(candidates)): # 剪枝，从当前位置开始，后续元素都比当前元素大
                if candidates[j] > left: # 剪枝
                    break
                path.append(candidates[j])
                dfs(j, left - candidates[j]) # 可重复选择，继续判断从当前范围开始选
                path.pop() # 此位置判断结束，恢复现场

        dfs(0, target)
        return res